In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 27. Flash Attentiontext text text

> **Learning Goals**
> - Flash Attentiontext text text(GPU SRAM vs HBM)text text text text
> - IO textdegrees text text(tiling)text text
> - text text text text text

## 27.1 GPU text text

GPUtext text text text:
- **SRAM** (text, text text, text ~20MB)
- **HBM** (High Bandwidth Memory, text "GPU text"text text, ~40-80GB)
- **DRAM** (CPU text, text, text)

HBM ↔ SRAM text text. Attentiontext $n \times n$ Matrixtext HBMtext text text → text.

## 27.2 text Attentiontext text text

$$S = QK^\top \in \mathbb{R}^{n \times n}$$  (HBMtext text)
$$P = \mathrm{softmax}(S) \in \mathbb{R}^{n \times n}$$  (HBM text text)
$$O = PV \in \mathbb{R}^{n \times d}$$  (HBM text text)

- HBM text: $O(n^2)$ (text)
- SRAM text: text text

## 27.3 Flash Attention — text Online Softmax

**text text**: $n \times n$ Matrixtext text text text, text text SRAMtext text.

### Online Softmax
text text text Calculationtext text text, text $\max$text $\sum$text text Calculation:
$$m_i^{(j)} = \max(m_i^{(j-1)}, \max_j S_{ij})$$
$$\ell_i^{(j)} = e^{m_i^{(j-1)} - m_i^{(j)}} \ell_i^{(j-1)} + \sum_j e^{S_{ij} - m_i^{(j)}}$$

### Tiling
$Q, K, V$text text text SRAMtext text, text Attention Calculation text Resulttext HBMtext text.

**IO textdegrees**:
- text: $O(n^2)$ HBM text
- Flash: $O(n^2 d / M)$ where $M$ = SRAM text → text text

### Backpropagation textCalculation
Backpropagation text $S, P$text text text textCalculation (text $O(n)$ vs $O(n^2)$).


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# text Attention
def standard_attention(Q, K, V):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-1, -2) / (d_k ** 0.5)  # HBM text
    P = F.softmax(S, dim=-1)  # HBM text/text
    O = P @ V  # HBM text/text
    return O

# Flash Attention text (text, online softmax)
def flash_attention_naive(Q, K, V, block_size=64):
    """Flash attentiontext text text text text."""
    B, H, N, D = Q.shape
    O = torch.zeros_like(Q)
    # Qtext Blocktext, K/Vtext text (text Flashtext text text Block)
    for i in range(0, N, block_size):
        Q_block = Q[:, :, i:i+block_size, :]  # (B, H, bs, D)
        S_block = Q_block @ K.transpose(-1, -2) / (D ** 0.5)  # (B, H, bs, N)
        P_block = F.softmax(S_block, dim=-1)
        O[:, :, i:i+block_size, :] = P_block @ V
    return O

# text text
torch.manual_seed(0)
B, H, N, D = 1, 2, 128, 32
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)

O_std = standard_attention(Q, K, V)
O_flash = flash_attention_naive(Q, K, V, block_size=32)
print(f"text vs Flash text Error: {(O_std - O_flash).abs().max():.2e}")
print("=> Resulttext text. Flashtext Memory Efficiencytext textLine.")


## 27.4 PyTorch SDPA text

PyTorchtext `scaled_dot_product_attention`text text text text text:
- `flash`: Flash Attention v2 (GPU)
- `mem_efficient`: Memory-Efficient Attention (GPU/CPU)
- `math`: text text


In [ ]:
# SDPA text Comparison
import time

# text text text text text
def bench_attention(n, d=64, n_heads=8, device='cpu'):
    """Attention Timetext Memory Measurement."""
    Q = torch.randn(1, n_heads, n, d, device=device)
    K = torch.randn(1, n_heads, n, d, device=device)
    V = torch.randn(1, n_heads, n, d, device=device)

    # Standard Implementation
    def std_attn():
        scores = Q @ K.transpose(-1, -2) / (d ** 0.5)
        weights = F.softmax(scores, dim=-1)
        return weights @ V

    # SDPA
    def sdpa_attn():
        return F.scaled_dot_product_attention(Q, K, V)

    # Time Measurement
    for _ in range(2): std_attn()  # warmup
    t0 = time.perf_counter()
    for _ in range(3):
        std_attn()
    t_std = (time.perf_counter() - t0) / 3 * 1000

    for _ in range(2): sdpa_attn()
    t0 = time.perf_counter()
    for _ in range(3):
        sdpa_attn()
    t_sdpa = (time.perf_counter() - t0) / 3 * 1000

    return t_std, t_sdpa

print(f"{'n':>6} | {'Standard (ms)':>15} | {'SDPA (ms)':>12} | {'Speedup':>10}")
print("-" * 55)
for n in [256, 512, 1024, 2048]:
    t_std, t_sdpa = bench_attention(n, device='cpu')
    print(f"{n:>6} | {t_std:>15.3f} | {t_sdpa:>12.3f} | {t_std/t_sdpa:>9.2f}x")


## 27.5 text text text

text Attentiontext $n \times n$ Attention Matrix text → $O(n^2)$ text.

Flash Attentiontext Attention Matrix text text text → $O(n)$ text.

text: $n = 8192, h = 32, d_k = 128, B = 1$:
- text: $B \cdot h \cdot n^2 \cdot 4 = 8$ GB (FP32)
- Flash: $O(n)$ → text 100MB


In [ ]:
# text text
def attention_memory_gb(n, n_heads, d_k, batch=1, dtype_bytes=4):
    """Standard Attentiontext Attention Matrix Memory."""
    return batch * n_heads * n * n * dtype_bytes / (1024**3)

print(f"Attention Matrix Memory (FP32):")
print(f"{'n':>6} | {'Memory (GB)':>12} | {'Flash (Estimation)':>14}")
print("-" * 40)
for n in [1024, 2048, 4096, 8192, 16384]:
    mem = attention_memory_gb(n, n_heads=32, d_k=128)
    flash_mem = n * 32 * 128 * 4 / (1024**3)  # text O(n)
    print(f"{n:>6} | {mem:>12.3f} | {flash_mem:>14.4f}")
print("\n=> n=8192text Standardtext 8GB, Flashtext text MB. Difference textdegreestext.")


## 27.6 Flash Attention v2/v3

- **v1** (Dao et al., 2022): text, online softmax
- **v2** (2023): text text text, Backpropagation text
- **v3** (2024): Hopper text (H100) text, text text

## 27.7 Ring Attention — text GPU text

$Q, K, V$text GPUtext text, GPU text text text Attention.
- text GPU text text text
- 1M+ text text text

## 27.8 Key Takeaways

| text | IO textdegrees | text | Speed |
|---|---|---|---|
| text Attention | $O(n^2)$ | $O(n^2)$ | text |
| Flash Attention v2 | $O(n^2 d/M)$ | $O(n)$ | text |
| Flash Attention v3 | text | $O(n)$ | H100text text |
| Ring Attention | text | $O(n/k)$ (k GPU) | text text text |

## Exercises

1. PyTorch SDPAtext text `math`, `mem_efficient`text text text Comparisontext.
2. Sequence Length 1024, 4096, 16384text text vs SDPA Timetext text Comparisontext.
3. Online Softmaxtext text text text text Resulttext text text.
4. Flash Attentiontext Backpropagation text text text text.
5. Ring Attentiontext text GPUtext text text text.

> Solutions: `solutions/ch27_solutions.ipynb`
